In [6]:
pip install yfinance pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [12]:
import yfinance as yf
import pandas as pd
import numpy as np

# ===================== CONFIG =====================

#SYMBOLS = "MSFT"(P)
#SYMBOL = "AAPL" (P)
#SYMBOL = "GOOGL" (p)
#SYMBOL = "TSLA" (P)
#SYMBOL = "VOO" (L)
#SYMBOL = "META"(L)
#SYMBOL = "AMZN" (P)
SYMBOL = "AMZN"
START = "2026-04-01"
END = "2026-04-05"

NEUTRAL_THRESHOLD = 0.0001     
STOP_LOSS_PERCENT = 0.03      
INITIAL_CASH = 10_000

# =================================================


def classify(change: float) -> str:
    if abs(change) <= NEUTRAL_THRESHOLD:
        return "neutral"
    return "positive" if change > 0 else "negative"


# ---------------- DATA ----------------

df = yf.download(
    SYMBOL,
    start=START,
    end=END,
    interval="1m",
    auto_adjust=True,
    progress=False
)

# Flatten columns (fixes Series ambiguity issue)
df.columns = df.columns.get_level_values(0)

# Clean data
df = df.dropna()

# ---------------- FEATURES ----------------

df["pct_change"] = df["Close"].pct_change()
df["state"] = df["pct_change"].apply(classify)

# ---------------- BACKTEST ----------------

cash = INITIAL_CASH
position = 0.0
entry_price = None

equity_curve = []
trade_log = []

for i in range(1, len(df)):
    row = df.iloc[i]
    prev = df.iloc[i - 1]

    price = float(row["Close"])
    state = row["state"]
    prev_state = prev["state"]

    date = row.name

    # BUY: neutral → positive
    if position == 0 and prev_state == "neutral" and state == "positive":
        position = cash / price
        entry_price = price
        cash = 0.0

        trade_log.append(
            {"Date": date, "Action": "BUY", "Price": price}
        )

    # STOP LOSS
    elif position > 0 and price <= entry_price * (1 - STOP_LOSS_PERCENT):
        cash = position * price
        position = 0.0
        entry_price = None

        trade_log.append(
            {"Date": date, "Action": "STOP", "Price": price}
        )

    # SELL: two negatives
    elif position > 0 and prev_state == "negative" and state == "negative":
        cash = position * price
        position = 0.0
        entry_price = None

        trade_log.append(
            {"Date": date, "Action": "SELL", "Price": price}
        )

    equity_curve.append(cash + position * price)

# ---------------- RESULTS ----------------

equity_curve = pd.Series(equity_curve, index=df.index[1:])
returns = equity_curve.pct_change().dropna()

final_equity = equity_curve.iloc[-1]
total_return = (final_equity / INITIAL_CASH - 1) * 100
max_drawdown = (equity_curve / equity_curve.cummax() - 1).min() * 100
sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0

print("====== BACKTEST RESULTS ======")
print(f"Final Equity: ${final_equity:,.2f}")
print(f"Total Return: {total_return:.2f}%")
print(f"Max Drawdown: {max_drawdown:.2f}%")
print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Number of Trades: {len(trade_log)}")

# ---------------- TRADES ----------------

trades = pd.DataFrame(trade_log)
print("\nTrade Log (first 10 rows):")
print(trades.head(10))


====== BACKTEST RESULTS ======
Final Equity: $10,182.15
Total Return: 1.82%
Max Drawdown: -1.01%
Sharpe Ratio: 0.73
Number of Trades: 79

Trade Log (first 10 rows):
                       Date Action       Price
0 2026-04-01 13:49:00+00:00    BUY  208.707794
1 2026-04-01 14:17:00+00:00   SELL  210.740005
2 2026-04-01 14:30:00+00:00    BUY  210.880005
3 2026-04-01 14:33:00+00:00   SELL  210.610001
4 2026-04-01 14:36:00+00:00    BUY  210.619995
5 2026-04-01 14:44:00+00:00   SELL  211.139999
6 2026-04-01 15:02:00+00:00    BUY  211.524994
7 2026-04-01 15:09:00+00:00   SELL  211.860199
8 2026-04-01 15:18:00+00:00    BUY  212.089996
9 2026-04-01 15:21:00+00:00   SELL  212.087494
